# Bird Identifier: training on Kaggle

Fine-tunes `convnext_nano` on 100 European bird species and exports it to ONNX for the browser.

**Before running:**
1. Upload `ml/dist/bird-identifier-data.zip` as a Kaggle Dataset (*Datasets → New Dataset*), then attach it to this notebook (*Add Input*).
2. *Settings → Accelerator*: **GPU P100** (or GPU T4).
3. *Settings → Internet*: **on** (needed to download the pretrained weights; requires a phone-verified Kaggle account).
4. Click **Save Version → Save & Run All** to train in the background (about 30–40 min), or run cells one by one.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Find the dataset and set up the code

In [ ]:
import glob, os, shutil
from pathlib import Path

matches = glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)
assert matches, "Dataset not found: attach the uploaded dataset via 'Add Input'"
DATA_DIR = Path(matches[0]).parent.parent
print("Data:", DATA_DIR)

# Copy the training code to a writable folder so `python -m src.train` works
WORK = Path("/kaggle/working/ml")
shutil.copytree(DATA_DIR / "src", WORK / "src", dirs_exist_ok=True)
os.chdir(WORK)

!pip install -q onnx onnxscript onnxruntime

## 2. Train

Each epoch prints training loss and validation accuracy (top-1 = the first guess is right,
top-5 = the right species is among the top 5 guesses). The best epoch is saved as `best.pt`.

In [ ]:
!python -m src.train --data-dir {DATA_DIR} --out-dir /kaggle/working/runs --epochs 12 --batch-size 64

## 3. Results

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

RUN = Path(sorted(glob.glob("/kaggle/working/runs/*/"))[-1])
hist = pd.read_csv(RUN / "history.csv")
metrics = json.loads((RUN / "metrics.json").read_text())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist.epoch, hist.train_loss, label="train"); ax1.plot(hist.epoch, hist.val_loss, label="val")
ax1.set(title="Loss", xlabel="epoch"); ax1.legend()
ax2.plot(hist.epoch, hist.val_top1, label="top-1"); ax2.plot(hist.epoch, hist.val_top5, label="top-5")
ax2.set(title="Validation accuracy", xlabel="epoch", ylim=(0, 1)); ax2.legend()
plt.show()

print(f"Test top-1: {metrics['test_top1']:.1%}   top-5: {metrics['test_top5']:.1%}   (best epoch {metrics['best_epoch']})")
display(pd.DataFrame(metrics["per_class"]).head(10).style.set_caption("Hardest species"))
display(pd.DataFrame(metrics["most_confused"]).head(10).style.set_caption("Most confused pairs"))

## 4. Export to ONNX for the browser

In [ ]:
!python -m src.export_onnx {RUN}/best.pt --out /kaggle/working/export
for name in ["metrics.json", "history.csv"]:
    shutil.copy(RUN / name, "/kaggle/working/export/")
shutil.make_archive("/kaggle/working/bird-model", "zip", "/kaggle/working/export")
print("Download bird-model.zip from the Output panel (it contains model.onnx, labels.json, config.json, metrics).")